# Revised black-box topic inference (per noun): model differences

Compares Macro-F1 and Accuracy between the three
models (smol, qwen, granite) for PG19 and PubMed. For each dataset it reports
the difference for every pair of models and the largest absolute difference
per metric.

Scores are read from `<dataset>_<model>_black_box_topic_results/metrics.json`.

In [1]:
import json
from itertools import combinations
from pathlib import Path

import pandas as pd

RESULTS_ROOT = Path(".")
OUTPUT_DIRECTORY = Path("black_box_topic_model_differences_results")

DATASETS = ("pg19", "pubmed")
DATASET_LABELS = {"pg19": "PG19", "pubmed": "PubMed"}
MODELS = ("smol", "qwen", "granite")

METRICS = {
    "Macro-F1": ("held_out_test", "macro_f1"),
    "Accuracy": ("held_out_test", "accuracy"),
}

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

## 1. Scores per dataset and model

In [2]:
def load_scores() -> pd.DataFrame:
    rows = []
    for dataset in DATASETS:
        for model in MODELS:
            metrics_path = (
                RESULTS_ROOT
                / f"{dataset}_{model}_black_box_topic_results"
                / "metrics.json"
            )
            with metrics_path.open(encoding="utf-8") as handle:
                metrics = json.load(handle)
            row = {"dataset": DATASET_LABELS[dataset], "model": model}
            for metric_name, (section, key) in METRICS.items():
                row[metric_name] = metrics[section][key]
            rows.append(row)
    return pd.DataFrame(rows)


scores = load_scores()
scores.to_csv(OUTPUT_DIRECTORY / "scores.csv", index=False)
scores.round(4)

  dataset    model  Macro-F1  Accuracy
0    PG19     smol    0.3673    0.3722
1    PG19     qwen    0.4237    0.4333
2    PG19  granite    0.4936    0.4944
3  PubMed     smol    0.4027    0.4000
4  PubMed     qwen    0.3144    0.3222
5  PubMed  granite    0.4334    0.4333

## 2. Pairwise differences

`difference = model_a - model_b`. A positive value means `model_a` scores
higher. `abs_difference` is used to rank the pairs.

In [3]:
def pairwise_differences(scores: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset, dataset_scores in scores.groupby("dataset", sort=False):
        by_model = dataset_scores.set_index("model")
        for model_a, model_b in combinations(MODELS, 2):
            for metric_name in METRICS:
                score_a = by_model.loc[model_a, metric_name]
                score_b = by_model.loc[model_b, metric_name]
                rows.append(
                    {
                        "dataset": dataset,
                        "metric": metric_name,
                        "model_a": model_a,
                        "model_b": model_b,
                        "score_a": score_a,
                        "score_b": score_b,
                        "difference": score_a - score_b,
                        "abs_difference": abs(score_a - score_b),
                    }
                )
    return pd.DataFrame(rows)


differences = pairwise_differences(scores)
differences.to_csv(OUTPUT_DIRECTORY / "pairwise_differences.csv", index=False)
differences.round(4)

   dataset    metric model_a  ... score_b  difference  abs_difference
0     PG19  Macro-F1    smol  ...  0.4237     -0.0565          0.0565
1     PG19  Accuracy    smol  ...  0.4333     -0.0611          0.0611
2     PG19  Macro-F1    smol  ...  0.4936     -0.1263          0.1263
3     PG19  Accuracy    smol  ...  0.4944     -0.1222          0.1222
4     PG19  Macro-F1    qwen  ...  0.4936     -0.0699          0.0699
5     PG19  Accuracy    qwen  ...  0.4944     -0.0611          0.0611
6   PubMed  Macro-F1    smol  ...  0.3144      0.0883          0.0883
7   PubMed  Accuracy    smol  ...  0.3222      0.0778          0.0778
8   PubMed  Macro-F1    smol  ...  0.4334     -0.0307          0.0307
9   PubMed  Accuracy    smol  ...  0.4333     -0.0333          0.0333
10  PubMed  Macro-F1    qwen  ...  0.4334     -0.1190          0.1190
11  PubMed  Accuracy    qwen  ...  0.4333     -0.1111          0.1111

[12 rows x 8 columns]

## 3. Highest difference per dataset and metric

In [4]:
highest = differences.loc[
    differences.groupby(["dataset", "metric"], sort=False)["abs_difference"].idxmax()
].reset_index(drop=True)
highest.to_csv(OUTPUT_DIRECTORY / "highest_differences.csv", index=False)

for row in highest.itertuples():
    better, worse = (
        (row.model_a, row.model_b) if row.difference >= 0 else (row.model_b, row.model_a)
    )
    print(
        f"{row.dataset:<7} {row.metric:<9} largest gap: {better} vs {worse} "
        f"= {row.abs_difference:.4f} ({row.abs_difference * 100:.2f} pp)"
    )

print(f"\nResults written to: {OUTPUT_DIRECTORY.resolve()}")
highest.round(4)

PG19    Macro-F1  largest gap: granite vs smol = 0.1263 (12.63 pp)
PG19    Accuracy  largest gap: granite vs smol = 0.1222 (12.22 pp)
PubMed  Macro-F1  largest gap: granite vs qwen = 0.1190 (11.90 pp)
PubMed  Accuracy  largest gap: granite vs qwen = 0.1111 (11.11 pp)

Results written to: /Users/marinabuitrago/Desktop/experiments-final/classificators/black-box-cat-final/no-cache/black_box_topic_model_differences_results


  dataset    metric model_a  ... score_b  difference  abs_difference
0    PG19  Macro-F1    smol  ...  0.4936     -0.1263          0.1263
1    PG19  Accuracy    smol  ...  0.4944     -0.1222          0.1222
2  PubMed  Macro-F1    qwen  ...  0.4334     -0.1190          0.1190
3  PubMed  Accuracy    qwen  ...  0.4333     -0.1111          0.1111

[4 rows x 8 columns]